# 3 - An?lise explorat?ria e Consultas SQL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import duckdb
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

### Carregando dados

In [ ]:
df = pd.read_parquet('arrests_limpo.parquet', engine='fastparquet')

df = df.replace('(null)', np.nan)

print("--- ESTADO INICIAL (ANTES DO TIDYING) ---")
df.info()
df.head(5)

## Transforma??es estruturais

In [ ]:
# N?o precisa.

## Normaliza??o

In [ ]:
print("Iniciando Padroniza??o (Pr?-requisitos)...")

print("Convertendo 'arrest_date' para datetime...")
df['arrest_date'] = pd.to_datetime(df['arrest_date'], errors='coerce')

print("Convertendo coordenadas para num?rico...")
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

print("Padroniza??o inicial conclu?da.")

print("Iniciando Normaliza??o...")

print("Separando 'arrest_date' em componentes...")
df['arrest_year'] = df['arrest_date'].dt.year
df['arrest_month'] = df['arrest_date'].dt.month
df['arrest_day'] = df['arrest_date'].dt.day
df['arrest_day_of_week'] = df['arrest_date'].dt.day_name()
df['arrest_hour'] = df['arrest_date'].dt.hour

print("Removendo coluna aninhada 'geocoded_column'...")
if 'geocoded_column' in df.columns:
    df = df.drop(columns=['geocoded_column'])

print("Normaliza??o conclu?da.")

## Padroniza??o de tipos

In [ ]:
print("Iniciando Padroniza??o (Otimiza??o)...")

print("Convertendo IDs (arrest_key, arrest_precinct) para string...")
df['arrest_key'] = df['arrest_key'].astype(str)
df['arrest_precinct'] = df['arrest_precinct'].astype(str)

print("Convertendo colunas de alta repeti??o para 'category'...")
colunas_categoricas = [
    'pd_cd', 'pd_desc', 'ky_cd', 'ofns_desc', 'law_code',
    'law_cat_cd', 'arrest_boro', 'jurisdiction_code',
    'age_group', 'perp_sex', 'perp_race', 'arrest_day_of_week'
]

for col in colunas_categoricas:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("Padroniza??o (Parte 2) conclu?da.")

print("--- ESTADO FINAL (DEPOIS DO TIDYING) ---")
df.info()

## Exporta??o de dados preparados em Parquet

In [ ]:
print("Exportando dados preparados...")
df.to_parquet('arrests_tidy_prepared.parquet', engine='fastparquet')
print("Arquivo 'arrests_tidy_prepared.parquet' salvo com sucesso!")

print("\nAmostra dos dados finais:")
df.head()

## Consultas SQL Anal?ticas

### --- 1. TEND?NCIA: M?dia M?vel de 7 dias ---

Pergunta: "Qual a m?dia m?vel de pris?es? As pris?es est?o aumentando ou diminuindo este ano?"

In [ ]:
con = duckdb.connect(database=':memory:')

sql_trend = """
WITH daily_counts AS (
    -- 1. CTE: Agrega o total de pris?es por dia
    SELECT
        CAST(arrest_date AS DATE) as date,
        COUNT(arrest_key) as total_arrests
    FROM df
    GROUP BY 1
)
-- 2. Window Function: Calcula a m?dia dos 7 dias anteriores
SELECT
    date,
    total_arrests,
    AVG(total_arrests) OVER (
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) as moving_avg_7_day
FROM daily_counts
ORDER BY date DESC
"""

# Rode e veja o resultado
trend_df = con.execute(sql_trend).df()
trend_df

### --- 2. COMPARA??O: Top 3 Crimes por Distrito ---

Pergunta: "Quais s?o os 3 principais tipos de crime (descri??o da ofensa) em cada Distrito (Borough)?"

In [ ]:
sql_top_crimes = """
WITH crime_counts AS (
    -- 1. CTE: Conta cada crime em cada distrito
    SELECT
        arrest_boro,
        ofns_desc, -- Usando a descri??o da ofensa
        COUNT(*) as crime_count
    FROM df
    WHERE arrest_boro IS NOT NULL AND ofns_desc IS NOT NULL
    GROUP BY 1, 2
),
ranked_crimes AS (
    -- 2. Window Function: Cria um ranking (1, 2, 3...)
    -- para cada crime DENTRO de cada distrito
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY arrest_boro -- Reinicia o ranking para cada distrito
            ORDER BY crime_count DESC -- O mais comum fica em 1?
        ) as rank
    FROM crime_counts
)
-- 3. Filtro Final: Pega s? o Top 3
SELECT
    arrest_boro,
    ofns_desc,
    crime_count,
    rank
FROM ranked_crimes
WHERE rank <= 3
ORDER BY arrest_boro, rank;
"""

top_crimes_df = con.execute(sql_top_crimes).df()
top_crimes_df

### --- 3. CONCENTRA??O: Delegacias "Hotspot" (Pareto) ---

Pergunta: "Quantas delegacias (precincts) s?o respons?veis por 50% de todas as pris?es? Onde o policiamento est? concentrado?"

In [ ]:
sql_concentration = """
WITH precinct_counts AS (
    -- 1. CTE: Conta o total por delegacia
    SELECT
        arrest_precinct,
        COUNT(*) as total_arrests
    FROM df
    GROUP BY 1
),
cumulative_percent AS (
    -- 2. CTE + Window Function: Calcula o % acumulado
    SELECT
        arrest_precinct,
        total_arrests,
        -- Soma o total de pris?es em ordem decrescente
        SUM(total_arrests) OVER (ORDER BY total_arrests DESC) as cumulative_arrests,
        -- Divide pelo total geral para ter o %
        100.0 * SUM(total_arrests) OVER (ORDER BY total_arrests DESC) / SUM(total_arrests) OVER () as cumulative_pct
    FROM precinct_counts
)
-- 3. Filtro Final: Mostra s? as delegacias que somam os primeiros 50%
SELECT
    arrest_precinct,
    total_arrests,
    cumulative_pct
FROM cumulative_percent
WHERE cumulative_pct <= 50  -- Mude para 80 se quiser o "Princ?pio de Pareto"
ORDER BY total_arrests DESC;
"""

concentration_df = con.execute(sql_concentration).df()
concentration_df

### --- 4. PADR?O: An?lise de Hora vs. Dia da Semana ---

Pergunta: "Pris?es por FELONY ASSAULT (Agress?o Grave) acontecem mais de dia ou de madrugada? E nos fins de semana?"

In [ ]:
sql_pattern = """
-- Criamos "faixas de hor?rio" para simplificar
WITH time_slots AS (
    SELECT
        *,
        CASE
            WHEN arrest_hour BETWEEN 6 AND 11 THEN '1. Manh? (06-11)'
            WHEN arrest_hour BETWEEN 12 AND 17 THEN '2. Tarde (12-17)'
            WHEN arrest_hour BETWEEN 18 AND 23 THEN '3. Noite (18-23)'
            ELSE '4. Madrugada (00-05)'
        END as time_of_day
    FROM df
    WHERE ofns_desc = 'FELONY ASSAULT' -- Filtre pelo crime que quer analisar
)
SELECT
    time_of_day,
    arrest_day_of_week,
    COUNT(*) as total_assault_arrests
FROM time_slots
WHERE arrest_day_of_week IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 3 DESC; -- Ordena pela faixa de hor?rio, depois pelo total
"""

pattern_df = con.execute(sql_pattern).df()
pattern_df

### --- 5. DEPEND?NCIA: Faixa Et?ria vs. Gravidade do Crime ---

Pergunta: "Existe uma rela??o vis?vel entre a faixa et?ria (age_group) e a gravidade do crime (law_cat_cd)?"

In [ ]:
sql_crosstab = """
SELECT
    age_group,
    -- Contagem para cada categoria de lei (usando PIVOT manual)
    COUNT(CASE WHEN law_cat_cd = 'F' THEN 1 END) as felony,
    COUNT(CASE WHEN law_cat_cd = 'M' THEN 1 END) as misdemeanor,
    COUNT(CASE WHEN law_cat_cd = 'V' THEN 1 END) as violation,
    COUNT(*) as total,
    
    -- % de pris?es daquele grupo que foram Felonies (o insight real)
    ROUND(100.0 * COUNT(CASE WHEN law_cat_cd = 'F' THEN 1 END) / COUNT(*), 2) as pct_felony
FROM df
WHERE age_group IS NOT NULL
GROUP BY 1
ORDER BY pct_felony DESC; -- Ordena por quem tem a maior % de crimes graves
"""

crosstab_df = con.execute(sql_crosstab).df()
crosstab_df

## An?lise Explorat?ria e Teste de Hip?teses

## 4.1 An?lise Univariada (Conhecendo as Pe?as)

Como os dados se distribuem em uma ?nica vari?vel?

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6) # Gr?ficos maiores

# (a) Vari?vel Categ?rica: 'arrest_boro' (Distrito)
# Vamos ver a distribui??o percentual de pris?es
print("--- Distribui??o de Pris?es por Distrito ---")
print(df['arrest_boro'].value_counts(normalize=True).mul(100).round(2).to_frame('% Total'))

# Visualiza??o
sns.countplot(data=df, x='arrest_boro', order=df['arrest_boro'].value_counts().index)
plt.title('Distribui??o de Pris?es por Distrito (Borough)')
plt.ylabel('Total de Pris?es')
plt.xlabel('Distrito')
plt.show()

# %%
# (b) Vari?vel Num?rica (Tidy): 'arrest_hour' (Hora da Pris?o)
# Isso mostra padr?es de "hora do rush" de pris?es
print("\n--- Distribui??o de Pris?es por Hora ---")

# Visualiza??o
sns.histplot(data=df, x='arrest_hour', bins=24, kde=True)
plt.title('Distribui??o de Pris?es por Hora do Dia')
plt.xlabel('Hora (0-23)')
plt.ylabel('Contagem')
plt.xticks(range(0, 24))
plt.show()

## 4.2 An?lise Bivariada e Teste de Hip?teses

### H1: Pessoas mais jovens cometem mais crimes que pessoas mais velhas.

In [ ]:
h1_sql = """
SELECT
    age_group,
    COUNT(arrest_key) as total_arrests
FROM df
WHERE age_group IS NOT NULL -- Ignora os nulos que limpamos
GROUP BY 1
ORDER BY 2 DESC
"""
h1_df = con.execute(h1_sql).df()
print(h1_df)

# Visualiza??o
sns.barplot(data=h1_df, x='age_group', y='total_arrests', order=h1_df.sort_values('total_arrests', ascending=False)['age_group'])
plt.title('H1: Total de Pris?es por Faixa Et?ria')
plt.ylabel('Total de Pris?es')
plt.xlabel('Faixa Et?ria')
plt.show()

### H2: Asi?ticos cometem menos crimes que outras etnias.

In [ ]:
h2_sql = """
SELECT
    perp_race,
    COUNT(arrest_key) as total_arrests
FROM df
WHERE perp_race IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC -- Ordena do maior para o menor
"""
h2_df = con.execute(h2_sql).df()
print(h2_df)

# Visualiza??o (gr?fico de barras horizontais ? melhor aqui)
sns.barplot(data=h2_df, y='perp_race', x='total_arrests', order=h2_df.sort_values('total_arrests', ascending=False)['perp_race'])
plt.title('H2: Total de Pris?es por Etnia')
plt.xlabel('Total de Pris?es')
plt.ylabel('Etnia (Perp. Race)')
plt.show()

### H3: Acontece mais crime no ver?o do que no inverno.

In [ ]:
h3_sql = """
-- Criamos 'temporada' (esta??o) para facilitar
WITH seasonal_arrests AS (
    SELECT
        arrest_key,
        arrest_month,
        CASE
            WHEN arrest_month IN (12, 1, 2) THEN '1. Inverno (Dez-Fev)'
            WHEN arrest_month IN (3, 4, 5) THEN '2. Primavera (Mar-Mai)'
            WHEN arrest_month IN (6, 7, 8) THEN '3. Ver?o (Jun-Ago)'
            WHEN arrest_month IN (9, 10, 11) THEN '4. Outono (Set-Nov)'
        END as season
    FROM df
    WHERE arrest_month IS NOT NULL
)
SELECT
    season,
    COUNT(arrest_key) as total_arrests
FROM seasonal_arrests
GROUP BY 1
ORDER BY 1 -- Ordena pelas esta??es
"""
h3_df = con.execute(h3_sql).df()
print(h3_df)

# Visualiza??o (Gr?fico de linha ? ?timo para sazonalidade)
sns.lineplot(data=h3_df, x='season', y='total_arrests', marker='o')
plt.title('H3: Sazonalidade das Pris?es (Hemisf?rio Norte)')
plt.ylabel('Total de Pris?es')
plt.xlabel('Esta??o do Ano')
plt.show()

## An?lise Multivariada

In [ ]:
crosstab_df = pd.crosstab(
    df['arrest_boro'],      # Linhas
    df['law_cat_cd'],       # Colunas
    normalize='index'       # Normaliza pela linha (mostra %)
)
crosstab_df = crosstab_df * 100 # Transforma em porcentagem (0-100)

print(crosstab_df.round(1))

# Visualiza??o: Heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(crosstab_df, annot=True, fmt='.1f', cmap='viridis')
plt.title('Heatmap: % de Gravidade do Crime por Distrito')
plt.ylabel('Distrito (Borough)')
plt.xlabel('Gravidade da Lei (F=Grave, M=M?dio, V=Leve)')
plt.show()